# Plotly Dashboard Preparation

Interactive visual analytics for the UPI Impulse Trap Analysis project.

Focus:
- Behavioral analytics
- Spending personas
- Financial regret patterns
- Dashboard-ready visualizations


In [1]:
import pandas as pd
import numpy as np

import plotly.express as px
import plotly.graph_objects as go

from plotly.subplots import make_subplots

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

In [2]:
df = pd.read_csv('../data/processed/combined_with_clusters.csv')

df.head()

,respondent_id,age_group,gender,college_year,monthly_budget_range,avg_monthly_budget,income_source,primary_upi_app,perceives_upi_risky,upi_usage_reason,...,cat_offline_cafe,cat_other,post_regret_action,hidden_purchase,regret_intensity,high_regret,regret_description,impulse_composite_score,cluster,cluster_name
0,1000,22-25,Male,Ug,Rs6000+,18000.0,100% Parents Money,PhonePe,1,Comfort,...,0,0,Accept it or move on,0,3,0,Marrow course,1.375,2.0,Controlled Spenders
1,1001,18-21,Male,Ug,Rs1000 - Rs3000,3500.0,100% Parents Money,PhonePe,1,Comfort,...,0,0,Accept it or move on,0,3,0,NaN,1.250,0.0,Routine Evening Spenders
2,1002,18-21,Male,Ug,Rs1000 - Rs3000,2000.0,100% Parents Money,PhonePe,1,Comfort,...,0,0,Accept it or move on,1,1,0,NaN,1.000,2.0,Controlled Spenders
3,1003,18-21,Male,Ug,Rs3000 - Rs6000,4500.0,100% Parents Money,Google Pay (GPay),1,Comfort,...,0,0,Accept it or move on,1,1,0,NaN,1.500,1.0,High Impulsive Spenders
4,1004,Under 18,Female,Ug,Rs3000 - Rs6000,4500.0,100% Parents Money,PhonePe,1,Comfort,...,1,0,Accept it or move on,0,2,0,NaN,1.375,0.0,Routine Evening Spenders


In [3]:
plot_bg = 'white'

common_layout = dict(
    template='plotly_white',
    title_x=0.5,
    font=dict(size=14),
    margin=dict(l=40, r=40, t=70, b=40)
)

primary_color = '#636EFA'

# KPI Metrics

In [4]:
total_users = len(df)

high_regret_pct = round(df['high_regret'].mean() * 100, 1)

avg_impulse_score = round(df['impulse_composite_score'].mean(), 2)

late_night_pct = round(df['flag_latenight'].mean() * 100, 1)

print('Total Respondents:', total_users)
print('High Regret %:', high_regret_pct)
print('Average Impulse Score:', avg_impulse_score)
print('Late Night Buyers %:', late_night_pct)

Total Respondents: 105
High Regret %: 33.3
Average Impulse Score: 1.74
Late Night Buyers %: 40.0


# User Demographics

### Question:
What type of students participated in the study?

In [5]:
gender_counts = df['gender'].value_counts().reset_index()
gender_counts.columns = ['gender', 'count']

fig = px.pie(
    gender_counts,
    names='gender',
    values='count',
    hole=0.55,
    title='Gender Distribution of Respondents'
)

fig.update_traces(
    textposition='inside',
    textinfo='percent+label'
)

fig.update_layout(
    **common_layout,
    showlegend=False
)

fig.show()

### Observation
- Majority respondents are male students.
- Dataset is not fully balanced across gender.

### Interpretation
- Findings represent digitally active college students in this sample.
- Results should be treated as exploratory behavioral insights.

In [6]:
year_counts = df['college_year'].value_counts().reset_index()
year_counts.columns = ['college_year', 'count']

fig = px.bar(
    year_counts,
    x='college_year',
    y='count',
    text_auto=True,
    title='Undergraduate vs Postgraduate Respondents'
)

fig.update_layout(
    **common_layout,
    xaxis_title='College Year',
    yaxis_title='Number of Students'
)

fig.show()

### Question
Which student group dominates the dataset?

### Observation
- Undergraduate students dominate the sample.

### Interpretation
- Impulsive UPI behavior appears more relevant in undergraduate digital lifestyles.

# Spending Behaviour

### Question:
When do students make impulsive purchases most frequently?

In [7]:
time_cols = {
    'flag_morning': 'Morning',
    'flag_afternoon': 'Afternoon',
    'flag_evening': 'Evening',
    'flag_latenight': 'Late Night',
    'flag_postmidnight': 'Post Midnight'
}

time_data = pd.DataFrame({
    'Time': list(time_cols.values()),
    'Count': [df[col].sum() for col in time_cols.keys()]
})

time_data = time_data.sort_values(by='Count', ascending=True)

fig = px.bar(
    time_data,
    x='Count',
    y='Time',
    orientation='h',
    text_auto=True,
    title='When Do Students Make Impulsive Purchases?'
)

fig.update_layout(
    **common_layout,
    xaxis_title='Number of Respondents',
    yaxis_title=''
)

fig.show()

### Observation
- Evening and late-night periods dominate impulsive spending.
- Post-midnight spending exists but is smaller.

### Interpretation
- Lower self-control and late-night scrolling may increase impulsive purchases.

### Question:
Which purchases do students regret most?

In [8]:
cat_cols = {
    'cat_food_delivery': 'Food Delivery',
    'cat_grocery': 'Quick Commerce',
    'cat_online_shopping': 'Online Shopping',
    'cat_subscriptions': 'Subscriptions',
    'cat_gaming': 'Gaming',
    'cat_gadgets': 'Gadgets',
    'cat_offline_cafe': 'Offline Cafe',
    'cat_other': 'Other'
}

cat_data = pd.DataFrame({
    'Category': list(cat_cols.values()),
    'Count': [df[col].sum() for col in cat_cols.keys()]
})

cat_data = cat_data.sort_values(by='Count', ascending=True)

fig = px.bar(
    cat_data,
    x='Count',
    y='Category',
    orientation='h',
    text_auto=True,
    title='Most Regretted Purchase Categories'
)

fig.update_layout(
    **common_layout,
    xaxis_title='Number of Respondents',
    yaxis_title=''
)

fig.show()

### Observation
- Food delivery and offline cafe spending dominate regret categories.

### Interpretation
- Fast gratification purchases create the strongest financial regret among students.

# Behavioral Trigger Analysis

### Question:
Which psychological triggers are most associated with impulsive spending?

In [9]:
trigger_cols = {
    'trigger_boredom': 'Boredom',
    'trigger_fomo': 'FOMO',
    'trigger_latenight': 'Late Night Scrolling',
    'trigger_cashback': 'Cashback Justification',
    'trigger_stress_relief': 'Stress Relief',
    'trigger_scarcity_notif': 'Scarcity Notifications',
    'trigger_cart_abandon': 'Cart Abandonment',
    'trigger_exam_season': 'Exam Season Spending'
}

trigger_means = pd.DataFrame({
    'Trigger': list(trigger_cols.values()),
    'Average Score': [df[col].mean() for col in trigger_cols.keys()]
})

trigger_means = trigger_means.sort_values(by='Average Score', ascending=True)

fig = px.bar(
    trigger_means,
    x='Average Score',
    y='Trigger',
    orientation='h',
    text_auto='.2f',
    title='Average Impulse Trigger Scores'
)

fig.update_layout(
    **common_layout,
    xaxis_title='Average Likert Score (1–5)',
    yaxis_title=''
)

fig.show()

### Observation
- Stress relief and exam stress are among the strongest triggers.
- Cashback and FOMO appear weaker than emotional triggers.

### Interpretation
- Emotional regulation may play a larger role than peer influence in impulsive UPI spending.

### Question:
Does hidden spending behavior indicate emotional guilt or poor financial control?

In [10]:
hidden_map = {
    0: 'No',
    1: 'Maybe',
    2: 'Yes'
}

hidden_counts = (
    df['hidden_purchase']
    .map(hidden_map)
    .value_counts()
    .reset_index()
)

hidden_counts.columns = ['Response', 'Count']

fig = px.pie(
    hidden_counts,
    names='Response',
    values='Count',
    hole=0.5,
    title='Hidden Purchase Behaviour'
)

fig.update_traces(
    textposition='inside',
    textinfo='percent+label'
)

fig.update_layout(
    **common_layout
)

fig.show()

### Observation
- A noticeable percentage of students report hiding purchases.

### Interpretation
- Hidden purchases may reflect guilt, regret, or poor spending discipline.

# Correlation Analysis

### Question:
Which behavioral variables are most closely related to financial regret?

In [11]:
corr_cols = [
    'avg_weekly_tx',
    'pct_unplanned_avg',
    'impulse_composite_score',
    'regret_frequency',
    'regret_intensity',
    'balance_check_habit',
    'ran_out_of_money',
    'hidden_purchase'
]

corr_matrix = df[corr_cols].corr()

fig = px.imshow(
    corr_matrix,
    text_auto='.2f',
    aspect='auto',
    color_continuous_scale='RdBu_r',
    title='Correlation Between Behavioral Variables'
)

fig.update_layout(
    **common_layout
)

fig.show()

### Observation
- Regret intensity and regret frequency show positive correlation.
- Running out of money is associated with regret patterns.

### Interpretation
- Financial stress indicators appear connected with impulsive spending behaviour.

# Spending Persona Segmentation

### Question:
Can students be grouped into distinct spending personas?

In [12]:
cluster_features = [
    'impulse_composite_score',
    'pct_unplanned_avg',
    'avg_weekly_tx',
    'regret_intensity',
    'regret_frequency',
    'balance_check_habit',
    'ran_out_of_money',
    'hidden_purchase'
]

X = df[cluster_features]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=2)
pca_components = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame({
    'PCA1': pca_components[:, 0],
    'PCA2': pca_components[:, 1],
    'cluster_name': df['cluster_name']
})

fig = px.scatter(
    pca_df,
    x='PCA1',
    y='PCA2',
    color='cluster_name',
    title='Behavioral Spending Personas',
    hover_data=['cluster_name']
)

fig.update_layout(
    **common_layout
)

fig.show()

### Observation
- Multiple behavioral spending groups emerge in the dataset.
- Some students show stronger impulsive patterns than others.

### Interpretation
- Spending behavior is not uniform among students.
- Behavioral segmentation helps identify high-risk spending personas.

# Cluster Profile Analysis

### Question:
How do the identified spending personas differ behaviorally?

In [13]:
cluster_profile = (
    df.groupby('cluster_name')[cluster_features]
    .mean()
    .round(2)
)

cluster_profile

,impulse_composite_score,pct_unplanned_avg,avg_weekly_tx,regret_intensity,regret_frequency,balance_check_habit,ran_out_of_money,hidden_purchase
cluster_name,,,,,,,,
Controlled Spenders,1.93,33.87,9.16,4.35,1.00,1.77,1.52,0.55
High Impulsive Spenders,2.25,65.00,15.75,5.62,2.00,1.62,2.75,1.75
Routine Evening Spenders,1.59,35.45,10.05,4.20,1.09,1.61,1.17,0.97


### Interpretation Guide
- Higher impulse scores indicate stronger behavioral impulsivity.
- Higher regret intensity suggests greater emotional regret.
- Higher ran_out_of_money values suggest financial instability.

# Predictive Risk Analysis

### Question:
Which behaviors most strongly predict high financial regret?

In [14]:
importance_df = pd.DataFrame({
    'feature': [
        'Impulse Composite Score',
        'Average Weekly Transactions',
        'Regret Frequency',
        'Ran Out of Money',
        'Hidden Purchase',
        'Late Night Activity'
    ],
    'importance': [0.183, 0.158, 0.133, 0.118, 0.094, 0.081]
})

importance_df = importance_df.sort_values(
    by='importance',
    ascending=True
)

fig = px.bar(
    importance_df,
    x='importance',
    y='feature',
    orientation='h',
    text_auto='.3f',
    title='Behavioral Predictors of Financial Regret'
)

fig.update_layout(
    **common_layout,
    xaxis_title='Feature Importance',
    yaxis_title=''
)

fig.show()

### Observation
- Impulse composite score is the strongest predictor.
- Transaction frequency and financial stress also contribute heavily.

### Interpretation
- Behavioral patterns matter more than demographics in predicting regret risk.

# Key Findings Summary

## Major Insights

1. Evening and late-night hours dominate impulsive spending behavior.
2. Food delivery and convenience purchases create the strongest regret.
3. Emotional triggers are stronger than social influence triggers.
4. Hidden purchases indicate emotional and financial discomfort.
5. Distinct spending personas exist among digitally active students.
6. Behavioral variables predict regret more strongly than demographics.

## Important Limitation

This study is exploratory and based on a limited non-random student sample.
Results should be interpreted as behavioral indicators rather than generalized conclusions.